In [1]:
import numpy as np
import pandas as pd
import os
import cv2
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, GlobalAveragePooling2D, Dropout,BatchNormalization, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

In [2]:
def image_import(path):
    Data=[]
    Labels=[]
    Categories=['cat','dog']
    for cate in Categories:
        folder=os.path.join(path,cate)
        label=Categories.index(cate)
        for img in os.listdir(folder):
            img_path=os.path.join(folder,img)
            image=cv2.imread(img_path)
            if image is None:
                continue
            image=cv2.resize(image,(128,128))
            Data.append(image)
            Labels.append(label)
    return np.array(Data),np.array(Labels) 

In [3]:
train_data='train'
val_data='val'


In [4]:
X_train,y_train=image_import(train_data)
X_val,y_val=image_import(val_data)


In [5]:
X_train = preprocess_input(X_train.astype('float32'))
X_val = preprocess_input(X_val.astype('float32'))


train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

Train_Datagen = train_datagen.flow(
    X_train, y_train,
    batch_size=32,
    shuffle=True
)
val_datagen = tf.keras.preprocessing.image.ImageDataGenerator()

Val_data = val_datagen.flow(
    X_val, y_val,
    batch_size=32,
    shuffle=False
)

In [6]:
vgg_base=VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)
for layer in vgg_base.layers:
    layer.trainable=False

In [7]:
model=Sequential([
    vgg_base,
    Conv2D(512, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    GlobalAveragePooling2D(),
    Dense(512,activation='relu'),
    Dropout(0.5),
    Dense(256,activation='relu'),
    Dropout(0.4),
    Dense(128,activation="relu"),
    Dropout(0.3),
    Dense(1,activation='sigmoid')   
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 4, 4, 512)         14714688  
                                                                 
 conv2d (Conv2D)             (None, 2, 2, 512)         2359808   
                                                                 
 global_average_pooling2d (  (None, 512)               0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 512)               262656    
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 256)               131328    
                                                        

In [8]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [9]:


reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

In [10]:
model.fit(
    Train_Datagen,
    epochs=50,
    validation_data=Val_data,
    callbacks=[ reduce_lr]
)

Epoch 1/50


9/9 [==============================] - 28s 3s/step - loss: 8.8425 - accuracy: 0.5345 - val_loss: 7.5732 - val_accuracy: 0.6571 - lr: 1.0000e-04
Epoch 2/50
9/9 [==============================] - 23s 3s/step - loss: 7.1729 - accuracy: 0.6909 - val_loss: 5.9988 - val_accuracy: 0.8143 - lr: 1.0000e-04
Epoch 3/50
9/9 [==============================] - 22s 3s/step - loss: 7.1630 - accuracy: 0.6545 - val_loss: 5.8436 - val_accuracy: 0.8286 - lr: 1.0000e-04
Epoch 4/50
9/9 [==============================] - 21s 2s/step - loss: 6.3038 - accuracy: 0.7127 - val_loss: 5.9928 - val_accuracy: 0.8143 - lr: 1.0000e-04
Epoch 5/50
9/9 [==============================] - 17s 2s/step - loss: 6.3353 - accuracy: 0.7418 - val_loss: 5.7698 - val_accuracy: 0.8429 - lr: 1.0000e-04
Epoch 6/50
9/9 [==============================] - 17s 2s/step - loss: 6.2022 - accuracy: 0.7709 - val_loss: 5.4671 - val_accuracy: 0.8857 - lr: 1.0000e-04
Epoch 7/50
9/9 [==============================] - 18s 2s/step - loss

In [11]:
from sklearn.metrics import confusion_matrix, classification_report
y_pred = (model.predict(X_val) > 0.5).astype(int)
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

3/3 [==============================] - 5s 1s/step
[[20  4]
 [ 0 46]]
              precision    recall  f1-score   support

           0       1.00      0.83      0.91        24
           1       0.92      1.00      0.96        46

    accuracy                           0.94        70
   macro avg       0.96      0.92      0.93        70
weighted avg       0.95      0.94      0.94        70

